<a href="https://colab.research.google.com/github/vishal9198/genAi-Labs/blob/main/hyde_technique.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 1. INSTALLATION & SETUP
# ==============================================================================
!pip install -qU langchain langchain-openai langchain-community langchain-chroma tiktoken beautifulsoup4

import os
from google.colab import userdata

# Automatically pull the key from Colab Secrets (Name: OPENAI_API_KEY)
try:
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter OpenAI API Key: ")

# ==============================================================================
# 2. VECTORSTORE INDEXING (Sample Corpus Setup)
# ==============================================================================
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# Load Lilian Weng's Agent post as our target knowledge base
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))
    )
)
blog_docs = loader.load()

# Split the blog post into ~300 token chunks with an overlap of 50 tokens
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)
splits = splitter.split_documents(blog_docs)

# Embed and index into an in-memory Chroma vectorstore
vectorstore = Chroma.from_documents(documents=splits, embedding=OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# ==============================================================================
# 3. HYDE PROMPT & HYPOTHETICAL DOCUMENT GENERATION
# ==============================================================================
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Instruct the LLM to write in the specific style of the target corpus (e.g., scientific paper passage)
template_hyde = """Please write a scientific paper passage to answer the question.
Question: {question}
Passage:"""

prompt_hyde = ChatPromptTemplate.from_template(template_hyde)
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# Build the Hypothetical Document Generator Chain:
# 1. Takes {"question": "..."}
# 2. Formats it into the template
# 3. LLM drafts a synthetic passage
# 4. Parses output into a raw string
generate_docs_for_retrieval = prompt_hyde | llm | StrOutputParser()

# Test question
question = "What is task decomposition for LLM agents?"

# Inspect the generated hypothetical passage
hypothetical_doc = generate_docs_for_retrieval.invoke({"question": question})
print("=== HYPOTHETICAL DOCUMENT DRAFTED BY LLM ===")
print(hypothetical_doc)
print("\n" + "=" * 60 + "\n")

# ==============================================================================
# 4. RETRIEVAL USING HYPOTHETICAL DOCUMENT
# ==============================================================================
# The pipeline pipes the hypothetical document string directly into the vectorstore retriever.
# The retriever embeds the passage and performs a vector search against the real documents.
retrieval_chain = generate_docs_for_retrieval | retriever

# Execute retrieval
retrieved_docs = retrieval_chain.invoke({"question": question})

print("=== ACTUAL DOCUMENTS RETRIEVED FROM VECTORSTORE ===")
for idx, doc in enumerate(retrieved_docs):
    print(f"[Document {idx + 1}]:\n{doc.page_content.strip()}\n")
print("=" * 60 + "\n")

# ==============================================================================
# 5. FINAL RAG SYNTHESIS (GROUNDED ANSWER)
# ==============================================================================
# Template for the final response, grounding the output in retrieved facts
template_rag = """Answer the following question based on this context:
{context}

Question: {question}
"""
prompt_rag = ChatPromptTemplate.from_template(template_rag)

# Helper function to unpack Document objects into a continuous context block
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# Construct final RAG execution graph
final_rag_chain = prompt_rag | llm | StrOutputParser()

# Synthesize response using the original question and the retrieved real docs
final_answer = final_rag_chain.invoke({
    "context": format_docs(retrieved_docs),
    "question": question
})

print("=== FINAL GROUNDED ANSWER ===")
print(final_answer)